# H3d — Retaliation

**H3d**  politicians who have been accused of lying are more likely to
subsequently accuse their accuser of lying in return.

### Why the obvious test does not work

The natural approach — count how often an accusation is followed by a reply — is
confounded by **burstiness**: pairs of MPs feud in bouts, and a bout produces
accusations in both directions regardless of who started.

The obvious fix, comparing the window *after* an accusation with the window
*before*, also fails. If B accused A shortly **before** A accused B, then A's
accusation is itself retaliation. Both windows contain retaliation with the roles
swapped, so the comparison is symmetric by construction and returns ~1 whether or
not retaliation exists. The same objection applies to a lead-versus-lag placebo:
mutual retaliation produces lag ≈ lead.

Within a dyad, retaliation and burstiness are indistinguishable.

### The test used here: who gets picked

Change the question from *when* to **whom**.

> Given that A makes an accusation, is the person they choose more likely to be
> someone who recently accused them?

Burstiness affects **whether** A accuses at all — an angry month means more
accusations against everyone. It does not affect **which** of A's possible
targets is chosen. So conditioning on "A accused someone on this day" removes the
confound, and what remains is target selection — which is what retaliation
actually means: not "I accuse more", but "I accuse *them* back".

**Design.** For each real accusation A→B we build a choice set: B (the one
actually chosen) plus a random sample of other people A accuses at other times.
Each candidate is flagged `provoked` if they accused A in the preceding window.
Conditional logit, one stratum per accusation, SEs clustered by accuser.

An odds ratio above 1 means: among the people A might have accused, the ones who
recently accused A were more likely to be picked.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys; sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

try:
    ConditionalLogit = sm.ConditionalLogit
except AttributeError:
    from statsmodels.discrete.conditional_models import ConditionalLogit

from lib import data, viz
viz.apply_style()

WINDOWS = [30, 90, 365]     # days within which a prior accusation counts
N_CONTROLS = 10             # control targets sampled per accusation
SEED = 20260811

rng = np.random.default_rng(SEED)

## 1. Events

In [ ]:
con = data.duck()

ev = con.execute("""
    SELECT country, date,
           accuser_speaker_id AS a,
           target_speaker_id  AS b,
           COALESCE(is_interjection, 0) AS is_interjection
    FROM accusations
    WHERE target_type = 'person'
      AND accuser_speaker_id IS NOT NULL
      AND target_speaker_id  IS NOT NULL
      AND accuser_speaker_id <> target_speaker_id
      AND date IS NOT NULL AND length(date) >= 10
""").df()

ev["date"] = pd.to_datetime(ev["date"].str[:10], errors="coerce")
ev = ev.dropna(subset=["date"]).reset_index(drop=True)

print(f"accusations with both people identified : {len(ev):,}")
print(f"distinct accusers                       : {ev['a'].nunique():,}")

## 2. Choice sets

For each accusation A→B: the chosen target B, plus up to `N_CONTROLS` other
people A accuses at other times. Accusers with only one distinct target are
dropped — they have no choice to model.

`provoked` = this candidate accused A within the window before this accusation.
It is computed identically for the chosen target and for the controls.

In [ ]:
# who each accuser ever targets
targets_of = {a: np.array(sorted(set(g["b"])))
              for a, g in ev.groupby("a")}

# when candidate X accused person A: key (X, A) -> sorted dates
accused_by = {k: np.sort(g["date"].values)
              for k, g in ev.groupby(["b", "a"])}


def provoked(cand, accuser, t, d):
    """Did `cand` accuse `accuser` in the d days before t?"""
    r = accused_by.get((accuser, cand))
    if r is None:
        return 0
    return int(((r < t) & (r >= t - d)).any())


def build_choice_sets(events, w):
    d = np.timedelta64(w, "D")
    obs, chosen, prov, accuser_col = [], [], [], []
    k = 0
    for a, b, t in zip(events["a"], events["b"], events["date"].values):
        pool = targets_of.get(a)
        if pool is None or len(pool) < 2:
            continue
        others = pool[pool != b]
        if len(others) == 0:
            continue
        n = min(N_CONTROLS, len(others))
        ctrls = rng.choice(others, size=n, replace=False)

        obs.append(k); chosen.append(1); accuser_col.append(a)
        prov.append(provoked(b, a, t, d))
        for c in ctrls:
            obs.append(k); chosen.append(0); accuser_col.append(a)
            prov.append(provoked(c, a, t, d))
        k += 1

    return pd.DataFrame({"obs": obs, "chosen": chosen,
                         "provoked": prov, "accuser": accuser_col})


cs = build_choice_sets(ev, 90)
print(f"choice sets (accusations modelled) : {cs['obs'].nunique():,}")
print(f"rows (chosen + controls)           : {len(cs):,}")

## 3. The result

First the descriptive version: among the targets actually chosen, how many had
recently accused the accuser — versus among the controls.

In [ ]:
desc = cs.groupby("chosen")["provoked"].agg(["mean", "sum", "size"])
desc.index = ["control targets", "chosen target"]
desc.columns = ["% provoked", "n provoked", "n"]
desc["% provoked"] = (desc["% provoked"] * 100).round(2)
print(desc.to_string())

lift = (desc.loc["chosen target", "% provoked"]
        / max(desc.loc["control targets", "% provoked"], 1e-9))
print(f"\nchosen targets are {lift:.1f}x more likely to have recently accused")
print("the accuser than the people they could have accused instead")

In [ ]:
def fit_selection(cs):
    keep = cs.groupby("obs")["provoked"].transform(lambda s: s.nunique() > 1)
    d = cs[keep]
    if d.empty or d["obs"].nunique() < 10:
        return None, d
    m = ConditionalLogit(d["chosen"], d[["provoked"]], groups=d["obs"]).fit(disp=False)
    return m, d


rows = []
for w in WINDOWS:
    c = build_choice_sets(ev, w)
    m, d = fit_selection(c)
    if m is None:
        rows.append({"window_days": w, "OR": np.nan, "lo": np.nan,
                     "hi": np.nan, "p": np.nan, "informative_sets": 0})
        continue
    b = m.params["provoked"]
    lo, hi = m.conf_int().loc["provoked"]
    rows.append({"window_days": w, "OR": np.exp(b), "lo": np.exp(lo),
                 "hi": np.exp(hi), "p": m.pvalues["provoked"],
                 "informative_sets": d["obs"].nunique()})

sel = pd.DataFrame(rows)
print(sel.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("\nOR > 1 : having recently accused A raises the odds of being A's target")
print("         -> retaliation. OR ~ 1 -> target choice is unrelated to")
print("         who attacked you.")
print("\nOnly choice sets containing BOTH a provoked and an unprovoked candidate")
print("carry information; `informative_sets` is how many of those there are.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ok = sel.dropna(subset=["OR"])
y = np.arange(len(ok))
ax.errorbar(ok["OR"], y, xerr=[ok["OR"] - ok["lo"], ok["hi"] - ok["OR"]],
            fmt="o", capsize=4, color="#c0603f", ms=7)
ax.axvline(1, color="black", lw=1)
ax.set_yticks(y)
ax.set_yticklabels([f"{int(d)} days" for d in ok["window_days"]])
ax.set_xlabel("odds of being chosen as target, if you recently accused them (95% CI)")
ax.set_title("Retaliation: do politicians accuse back the person who accused them?")
fig.tight_layout()
viz.savefig(fig, "h3d_target_selection")
plt.show()

## 4. Robustness

In [ ]:
# (a) excluding interjections: heckles are attributed structurally, so the
#     accuser-target link is assumed rather than observed
ev_ni = ev[ev["is_interjection"] == 0]
targets_all, accused_all = targets_of, accused_by
targets_of = {a: np.array(sorted(set(g["b"]))) for a, g in ev_ni.groupby("a")}
accused_by = {k: np.sort(g["date"].values) for k, g in ev_ni.groupby(["b", "a"])}
m_ni, d_ni = fit_selection(build_choice_sets(ev_ni, 90))
targets_of, accused_by = targets_all, accused_all

if m_ni is not None:
    b = m_ni.params["provoked"]; lo, hi = m_ni.conf_int().loc["provoked"]
    print(f"excluding interjections : OR {np.exp(b):.3f} "
          f"[{np.exp(lo):.3f}, {np.exp(hi):.3f}]  p={m_ni.pvalues['provoked']:.3g}"
          f"  ({d_ni['obs'].nunique():,} sets)")

# (b) more controls per accusation -- checks the sampling is not doing the work
N_CONTROLS_SAVED = N_CONTROLS
for n_c in (5, 20):
    N_CONTROLS = n_c
    m_c, d_c = fit_selection(build_choice_sets(ev, 90))
    if m_c is not None:
        print(f"{n_c:>3} controls per set     : "
              f"OR {np.exp(m_c.params['provoked']):.3f}  "
              f"({d_c['obs'].nunique():,} sets)")
N_CONTROLS = N_CONTROLS_SAVED

In [ ]:
# (c) by country -- is one parliament driving it?
rows = []
for c, g in ev.groupby("country"):
    if len(g) < 1000:
        continue
    m_c, d_c = fit_selection(build_choice_sets(g, 90))
    if m_c is None:
        continue
    rows.append({"country": c, "n_events": len(g),
                 "sets": d_c["obs"].nunique(),
                 "OR": np.exp(m_c.params["provoked"]),
                 "p": m_c.pvalues["provoked"]})
byc = pd.DataFrame(rows).sort_values("OR", ascending=False)
print("90-day window, countries with >= 1000 events\n")
print(byc.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

## 5. Verdict

| check | OR | p |
|---|---|---|
| 30 days | | |
| 90 days | | |
| 365 days | | |
| excluding interjections | | |
| country spread | | |

**H3d supported** if the odds ratio is above 1: among the people an accuser could
have targeted, those who recently accused them are more likely to be chosen.

Why this identifies retaliation where the earlier attempts did not: every
candidate in a choice set shares the same accuser at the same moment, so the
accuser's mood, activity level and general antagonism are held fixed by
construction. Burstiness cannot explain which candidate is picked.

The remaining assumption is the control pool — other people this accuser targets
at other times. It holds the accuser fixed but not the *opportunity* to accuse
each candidate on that particular day. A candidate absent from the chamber that
week could not have been targeted, which adds noise but is unlikely to correlate
with having recently accused the accuser.